## Training with classic machine learning models

In [ ]:
import pandas as pd

snapchat_df = pd.read_json("labeled_snapchat_comparison.jsonl", lines=True)
youtube_df = pd.read_json("labeled_youtube_comparison.jsonl", lines=True)

snapchat_df["source"] = "snapchat"
youtube_df["source"] = "youtube"

df = pd.concat([snapchat_df, youtube_df], ignore_index=True)

# Flatten the body text
df["text"] = df["body"].apply(lambda x: " ".join(x))

# Use LLM for ground truth
df["label"] = df["llm_sentiment"].str.lower().str.strip()

# Map to integers
label_map = {"negative": 0, "neutral": 1, "positive": 2}
df["label_id"] = df["label"].map(label_map)

# Drop NaNs or unknowns
df = df.dropna(subset=["label_id", "text"])
print(df['label'].value_counts())

print(df.head())

label
negative    1364
neutral      472
positive     166
Name: count, dtype: int64
                                               title  \
0    Should Indonesia ban social media for children?   
1  Should Indonesia ban social media for children...   
2  Social Media's Grip on Teen Lives: A Deep Dive...   
3  Australia: Under-16s banned from social media ...   
4                     Breaking the Doom-Scroll Cycle   

                                                body  \
0  [S, ocial media addiction is becoming an incre...   
1  [S, ocial media addiction is becoming an incre...   
2  [A recent report by the Pew Research Centre hi...   
3  [From November 2025, Australians under the age...   
4  [This article is written by a student writer f...   

                                            provider  \
0  {'name': 'thejakartapost.com', 'domain': 'thej...   
1  {'name': 'thejakartapost.com', 'domain': 'thej...   
2  {'name': 'devdiscourse.com', 'domain': 'devdis...   
3  {'name': 'sxm-ta

## data cleaning

In [ ]:
import string
import re

In [ ]:

def cleaning(text):        
    # converting to lowercase, removing URL links, special characters, punctuations...
    text = text.lower() # converting to lowercase
    text = re.sub(r"\b\d+\b", "", text) # removing number 
    text = re.sub('<.*?>+', '', text) # removing special characters, 
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text) # punctuations
    text = re.sub('\n', '', text)
    text = re.sub('[’“”…]', '', text)   

    return text
    
df["cleantext"] = df['text'].apply(cleaning)

## split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X, y_train, y = train_test_split(
    df["cleantext"], df["label_id"], test_size=0.3, random_state=42, stratify=df["label_id"]
)

X_test, X_val, y_test, y_val = train_test_split(
    X, y, test_size=0.5, random_state=42, stratify=y
)



## Logistic Regression with TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=300)
clf.fit(X_train_tfidf, y_train)

y_pred = clf.predict(X_test_tfidf)
print(classification_report(y_test, y_pred, target_names=label_map.keys()))


              precision    recall  f1-score   support

    negative       0.90      0.94      0.92       204
     neutral       0.76      0.77      0.77        71
    positive       1.00      0.56      0.72        25

    accuracy                           0.87       300
   macro avg       0.89      0.76      0.80       300
weighted avg       0.87      0.87      0.87       300



## Transformer-based models

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

# Build Dataset object
train_df = pd.DataFrame({"text": X_train, "label": y_train})
test_df = pd.DataFrame({"text": X_val, "label": y_val})
dataset = Dataset.from_pandas(pd.concat([train_df, test_df]))
dataset = dataset.train_test_split(test_size=0.2)

# Tokenize
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
dataset = dataset.map(lambda e: tokenizer(e["text"], truncation=True, padding="max_length", max_length=256), batched=True)

# Model
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

training_args = TrainingArguments(
    output_dir="./bert_sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
)

bert_trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
)

bert_trainer.train()


/home/ys/Documents/584NLP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-09 03:36:56.337458: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-09 03:36:56.637440: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-09 03:36:57.686792: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on.

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.311300,0.388166,0.850440,0.586627
2,0.367500,0.290764,0.912023,0.846051
3,0.158300,0.362797,0.914956,0.846704
4,0.084100,0.363301,0.926686,0.877379
5,0.055100,0.410033,0.935484,0.892216
6,0.016200,0.418668,0.929619,0.883283
7,0.095500,0.443929,0.932551,0.885703
8,0.108100,0.417724,0.935484,0.894856
9,0.002800,0.427105,0.935484,0.893922
10,0.000800,0.427984,0.935484,0.893922


TrainOutput(global_step=1710, training_loss=0.11694307988645694, metrics={'train_runtime': 325.2948, 'train_samples_per_second': 41.839, 'train_steps_per_second': 5.257, 'total_flos': 1790486807639040.0, 'train_loss': 0.11694307988645694, 'epoch': 10.0})

In [ ]:
# Test models
testset = pd.DataFrame({"text": X_test, "label": y_test})
testset = Dataset.from_pandas(testset)
testset = testset.map(lambda e: tokenizer(e["text"], truncation=True, padding="max_length", max_length=256), batched=True)

test = bert_trainer.evaluate(testset)
print("\nBERT Test metrics:", test)

Map: 100%|██████████| 300/300 [00:00<00:00, 3499.86 examples/s]



BERT Test metrics: {'eval_loss': 0.5255650281906128, 'eval_accuracy': 0.9, 'eval_f1_macro': 0.8448138247506733, 'eval_runtime': 1.9175, 'eval_samples_per_second': 156.452, 'eval_steps_per_second': 19.817, 'epoch': 10.0}


## RoBERTa

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import evaluate


tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")
model = AutoModelForSequenceClassification.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
# Build Dataset object
train_df = pd.DataFrame({"text": X_train, "label": y_train})
test_df = pd.DataFrame({"text": X_val, "label": y_val})
dataset = Dataset.from_pandas(pd.concat([train_df, test_df]))
dataset = dataset.train_test_split(test_size=0.2)

# Tokenize
dataset = dataset.map(lambda e: tokenizer(e["text"], truncation=True, padding="max_length", max_length=256), batched=True)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

training_args = TrainingArguments(
    output_dir="./RoBERTa_sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
)

trainer.train()


Map: 100%|██████████| 341/341 [00:00<00:00, 5207.53 examples/s]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.413800,0.359266,0.900293,0.818782
2,0.255400,0.431127,0.906158,0.821228
3,0.154200,0.514614,0.912023,0.826904
4,0.163100,0.601592,0.909091,0.813677
5,0.005200,0.524125,0.909091,0.819718
6,0.116400,0.558583,0.912023,0.832012
7,0.080100,0.536203,0.917889,0.842521
8,0.003100,0.578887,0.914956,0.836299
9,0.001500,0.566116,0.917889,0.842240
10,0.001100,0.567169,0.917889,0.842240


TrainOutput(global_step=1710, training_loss=0.1037195564488668, metrics={'train_runtime': 331.6036, 'train_samples_per_second': 41.043, 'train_steps_per_second': 5.157, 'total_flos': 1790486807639040.0, 'train_loss': 0.1037195564488668, 'epoch': 10.0})

In [ ]:
# Test models
testset = pd.DataFrame({"text": X_test, "label": y_test})
testset = Dataset.from_pandas(testset)
testset = testset.map(lambda e: tokenizer(e["text"], truncation=True, padding="max_length", max_length=256), batched=True)

Ro_test = trainer.evaluate(testset)
print("\nRoBERTa Test metrics:", Ro_test)

Map: 100%|██████████| 300/300 [00:00<00:00, 4322.44 examples/s]



RoBERTa Test metrics: {'eval_loss': 0.6866369843482971, 'eval_accuracy': 0.8933333333333333, 'eval_f1_macro': 0.8336053593309858, 'eval_runtime': 1.9295, 'eval_samples_per_second': 155.478, 'eval_steps_per_second': 19.694, 'epoch': 10.0}
